In [ ]:
!pip install scikit-learn

In [ ]:
# GATE TF PREDICTION ENGINE
# WITH NORMALIZATION
# ==========================================

import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from scipy import stats

# ==========================================
# LOAD CSV FILES
# ==========================================

frequency_path = r"D:\TF_Gate_Topic_predictor\data\topic_frequency.csv"

trend_path = r"D:\TF_Gate_Topic_predictor\data\trend_analysis.csv"

freq_df = pd.read_csv(frequency_path)

trend_df = pd.read_csv(trend_path)

# ==========================================
# TOTAL TOPIC FREQUENCY
# ==========================================

total_frequency = freq_df.groupby("Topic")[
    "Frequency"
].sum()

# ==========================================
# RECENT YEARS ANALYSIS
# ==========================================

recent_years = ["2023", "2022", "2021","2024", "2025", "2026"]

recent_df = freq_df[
    freq_df["Year"].astype(str).isin(recent_years)
]

recent_frequency = recent_df.groupby("Topic")[
    "Frequency"
].sum()

# ==========================================
# STORE PREDICTION RESULTS
# ==========================================

prediction_results = []

# ==========================================
# CALCULATE RAW SCORES
# ==========================================

for topic in total_frequency.index:

    # --------------------------------------
    # TOTAL FREQUENCY
    # --------------------------------------

    freq_score = total_frequency[topic]

    # --------------------------------------
    # TREND SCORE
    # --------------------------------------

    trend_row = trend_df[
        trend_df["Topic"] == topic
    ]

    
    if not trend_row.empty:

        
        trend_score = trend_row[
            "Trend"
        ].values[0]

    else:

        trend_score = 0

    # --------------------------------------
    # RECENT YEARS SCORE
    # --------------------------------------

    if topic in recent_frequency.index:

        recent_score = recent_frequency[topic]

    else:

        recent_score = 0

    # --------------------------------------
    # STORE RAW VALUES
    # --------------------------------------

    prediction_results.append({

        "Topic": topic,

        "Total Frequency": freq_score,

        "Trend Score": trend_score,

        "Recent Weight": recent_score

    })

# ==========================================
# CREATE DATAFRAME
# ==========================================

prediction_df = pd.DataFrame(
    prediction_results
)

# ==========================================
# NORMALIZATION
# ==========================================

scaler = MinMaxScaler()

columns_to_normalize = [

    "Total Frequency",

    "Trend Score",

    "Recent Weight"

]

prediction_df[columns_to_normalize] = scaler.fit_transform(

    prediction_df[columns_to_normalize]

)

# ==========================================
# FINAL PREDICTION SCORE
# ==========================================

prediction_df["Prediction Score"] = (

    prediction_df["Total Frequency"] * 0.2 +

    prediction_df["Trend Score"] * 0.1 +

    prediction_df["Recent Weight"] * 0.7

)

# ==========================================
# ROUND VALUES
# ==========================================

prediction_df = prediction_df.round(3)

# ==========================================
# SORT BY PREDICTION SCORE
# ==========================================

prediction_df = prediction_df.sort_values(

    by="Prediction Score",

    ascending=False

)

# ==========================================
# DISPLAY RESULTS
# ==========================================

print("\n===================================")
print("PREDICTED HIGH-WEIGHTAGE TOPICS")
print("===================================")

print(prediction_df)

# ==========================================
# TOP 5 PREDICTED TOPICS
# ==========================================

print("\n===================================")
print("TOP 5 PREDICTED TOPICS")
print("===================================")

top_5 = prediction_df.head(5)

for i, row in enumerate(top_5.values, start=1):

    print(f"{i}. {row[0]}")

# ==========================================
# SAVE CSV
# ==========================================

output_path = r"D:\TF_Gate_Topic_predictor\data\predicted_topics.csv"

prediction_df.to_csv(output_path, index=False)

# ==========================================
# FINISHED
# ==========================================

print("\n===================================")
print("PREDICTION FILE CREATED!")
print("===================================")

print(f"\nSaved at:\n{output_path}")
